# 06.11 — Single total loss → three subnetworks

[06.1](06.1_multi_task_losses_overview.ipynb) established it as a rule (Critical Note #28): the four loss components collapse into **one scalar**, and you call `.backward()` **once**. This notebook shows *why that works* — how a single scalar's gradient correctly trains three different subnetworks (encoder, decoder, classifier), giving the shared encoder the *sum* of every component's pull while each branch gets only its own. It's not a trick; it's what the autograd graph does for free once you wire it right.

This notebook covers:

* The topology: a shared encoder feeding a decoder branch and a classifier branch.
* `Loss_Encoder = Loss_Decoder + Loss_Classifier` — the assembly.
* How one `.backward()` *routes* each component to the right subnetwork.
* How the shared encoder *accumulates* both branches, tensor-for-tensor.

**Prerequisites:** [02.5 autograd](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb), [06.1 overview](06.1_multi_task_losses_overview.ipynb).

## Section 1 — What MATLAB does

`cgg_lossComponents` assembles the total from subnetwork sub-totals, then takes *one* gradient:

```matlab
Loss_Decoder    = Loss_Recon + Loss_KL + Loss_OffsetScale;   % decoder branch
Loss_Classifier = Loss_Class + Loss_Confidence;              % classifier branch
Loss_Encoder    = Loss_Decoder + Loss_Classifier;            % the single root

[grads] = dlgradient(Loss_Encoder, net.Learnables);          % ONE backward
```

The `Loss_Decoder` / `Loss_Classifier` split is **bookkeeping** — it lets MATLAB log each subnetwork's contribution. But there is a single gradient call, on `Loss_Encoder`. The port mirrors this exactly: `aggregate_normalized_losses` returns `total`, `decoder`, and `classifier`, and the training loop backprops only `total`.

## Section 2 — The Python concepts you need

### 2.1 — The topology: one encoder, two branches

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(11, 4.5))
def box(x, y, label, color, w=1.9, h=0.7):
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
        boxstyle="round,pad=0.05", facecolor=color, edgecolor="black", lw=1.3))
    ax.text(x, y, label, ha="center", va="center", fontsize=9.5, fontweight="bold")
def arr(x1, y1, x2, y2, c="black"):
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", color=c, lw=1.5))

box(1.2, 2.5, "ENCODER\n(shared)", "#cce4ff")
box(4.0, 2.5, "latent z", "#ffe4cc", w=1.3)
box(7.0, 3.7, "DECODER", "#e6f0d0")
box(7.0, 1.3, "CLASSIFIER", "#f0d0e6")
box(10.0, 3.7, "recon + KL\n+ offset", "#d0d0d0", w=1.9)
box(10.0, 1.3, "class + conf", "#d0d0d0", w=1.9)
arr(2.15, 2.5, 3.35, 2.5)
arr(4.65, 2.7, 6.05, 3.6); arr(4.65, 2.3, 6.05, 1.4)
arr(7.95, 3.7, 9.05, 3.7); arr(7.95, 1.3, 9.05, 1.3)
ax.text(10.0, 4.6, "Loss_Decoder", ha="center", fontsize=9, color="#4a7a1a", fontweight="bold")
ax.text(10.0, 0.4, "Loss_Classifier", ha="center", fontsize=9, color="#a01a7a", fontweight="bold")
ax.text(5.6, 0.1, "total = Loss_Decoder + Loss_Classifier  → ONE .backward()", ha="center", fontsize=10)
ax.set_xlim(0, 11.5); ax.set_ylim(-0.2, 5); ax.axis("off")
plt.tight_layout(); plt.show()
print("The encoder feeds BOTH branches, so its gradient comes from BOTH loss groups.")

The encoder is **shared**: its latent `z` feeds the decoder (reconstruction) *and* the classifier. The decoder and classifier are *not* shared — each sits on its own branch. This structure is the whole story: when gradients flow back, the branches get only their own component, but the encoder sits downstream of everything and collects it all.

### 2.2 — One scalar total loss

`total = Loss_Decoder + Loss_Classifier`, where each is a sum of its components. Build a faithful toy and confirm `total` is a single 0-D scalar:

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
encoder    = nn.Linear(4, 3)      # shared trunk
decoder    = nn.Linear(3, 4)      # reconstruction branch
classifier = nn.Linear(3, 2)      # classification branch

x = torch.randn(5, 4)
z = encoder(x)                                   # shared latent
loss_decoder    = ((decoder(z) - x) ** 2).mean()  # stands in for recon+KL+offset
loss_classifier = classifier(z).abs().sum()       # stands in for class+conf
total = loss_decoder + loss_classifier            # ONE scalar
print("total loss:", round(total.item(), 4), " shape:", tuple(total.shape), "(0-D scalar)")

### 2.3 — One `.backward()` routes each component to its subnetwork

Call `.backward()` once on `total`. Autograd walks the graph and deposits a gradient in every parameter that contributed. The routing falls out of the topology — the decoder only touched `loss_decoder`, so that's all it sees:

In [ ]:
total.backward()

def grad_norm(module):
    return sum(p.grad.abs().sum().item() for p in module.parameters() if p.grad is not None)

print(f"encoder    grad: {grad_norm(encoder):.4f}   ← from BOTH branches (shared trunk)")
print(f"decoder    grad: {grad_norm(decoder):.4f}   ← from reconstruction only")
print(f"classifier grad: {grad_norm(classifier):.4f}   ← from classification only")
print("One backward, three subnetworks, each trained by exactly the losses it produced.")

No manual routing, no three optimizers, no assigning "this loss trains that layer." The autograd graph *is* the routing table: a parameter receives gradient from a loss if and only if there's a path between them. The decoder has no path to `loss_classifier`, so it gets zero from it.

### 2.4 — The shared encoder *accumulates* both branches

The interesting parameter is the encoder — it's downstream of both losses. Autograd **sums** the two contributions into its `.grad`. Prove it at the tensor level (gradient *tensors* add; their norms don't, so we compare the tensors directly):

In [ ]:
# Single backward already done above — capture the encoder's combined gradient:
g_both = encoder.weight.grad.clone()

# Now isolate each branch's contribution with separate backwards:
encoder.zero_grad(); z = encoder(x); ((decoder(z) - x) ** 2).mean().backward()
g_from_decoder = encoder.weight.grad.clone()
encoder.zero_grad(); z = encoder(x); classifier(z).abs().sum().backward()
g_from_classifier = encoder.weight.grad.clone()

diff = (g_both - (g_from_decoder + g_from_classifier)).abs().max().item()
print(f"max | g_both − (g_decoder + g_classifier) | = {diff:.2e}")
print("→ ~0: the single backward deposits recon-grad + class-grad into the encoder, element-wise.")

That element-wise sum is *exactly why* you use one total loss instead of training the branches separately. The encoder needs the **combined** signal — "improve the latent so it both reconstructs well *and* classifies well" — and a single `.backward()` on the sum delivers precisely that, because $\nabla(a + b) = \nabla a + \nabla b$. The decoder and classifier, having no shared path, are unaffected by each other.

### 2.5 — Why not three separate backward passes?

You *could* call `.backward()` on each component. You shouldn't (Critical Note #28):

* **The shared encoder.** Three backwards would each try to backprop through the encoder. Without `retain_graph=True` the second call errors (the graph was freed); with it, you recompute the encoder's backward three times — wasteful, and you must be careful not to `zero_grad` between them or you lose the accumulation.
* **The single graph is free.** `total.backward()` traverses the shared encoder *once* and accumulates — no `retain_graph`, no recomputation. It's both simpler and faster.
* **Normalization couples them.** The components are already balanced against each other by the EMA normalization ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)) *before* summing — they're meant to travel together as one objective, not as three independent losses.

One scalar, one backward. The subnetwork decomposition (`decoder`, `classifier`) is returned for *logging*, not for separate optimization.

## Section 3 — The `neural_data_decoding` implementation

The assembly in `aggregate_normalized_losses` — three sub-totals, one root:

In [ ]:
from pathlib import Path
src = Path("../../src/neural_data_decoding/training/losses/multi_objective.py").read_text().splitlines()
i = next(j for j, line in enumerate(src) if "Assembly: Loss_Decoder" in line)
for k in range(i, i + 18):
    print(f"{k + 1:4} | {src[k]}")

Things to spot:

* `loss_decoder = recon + KL + offset/scale`; `loss_classifier = classification + confidence` — the two branch sub-totals (§2.2), each a `torch.stack(...).sum()` over its present components.
* `loss_encoder = loss_decoder + loss_classifier` — the single root, returned as `total`. This is the *only* tensor the training loop calls `.backward()` on.
* The `[p for p in (...) if p is not None]` filters absent components — a pure autoencoder ([05.6](../05_training_loop/05.6_the_two_stage_lifecycle.ipynb) Stage 1) has no `classifier` sub-total, so `total` is just the decoder loss and the classifier gets no gradient (nothing flows to it).
* `decoder` and `classifier` come back on the `NormalizedLossBreakdown` for *monitoring* — the loop logs them but backprops `total`. The naming mirrors MATLAB's `Loss_Encoder`/`Loss_Decoder`/`Loss_Classifier` bookkeeping (§1).

## Section 4 — Hands-on exercises

### Exercise 4.1 — a frozen branch gets no gradient

Detach the classifier's input (simulating a frozen/absent classifier branch) and show the classifier gets zero gradient while the decoder path still trains the encoder.

In [ ]:
# Reveal:
for m in (encoder, decoder, classifier):
    m.zero_grad()
z = encoder(x)
loss_dec = ((decoder(z) - x) ** 2).mean()
loss_cls = classifier(z.detach()).abs().sum()   # detach → no path back to encoder
(loss_dec + loss_cls).backward()
print(f"encoder grad (decoder path only reaches it): {grad_norm(encoder):.4f}  (nonzero)")
print(f"classifier grad (still trains ITSELF):       {grad_norm(classifier):.4f}  (nonzero)")
# but the encoder got NOTHING from the classifier branch because z was detached:
encoder.zero_grad(); z2 = encoder(x); classifier(z2.detach()).abs().sum().backward()
print(f"encoder grad from the DETACHED classifier path: {grad_norm(encoder):.4f}  (zero — detach cut it)")

### Exercise 4.2 — linearity of the gradient

Confirm that scaling one component's weight scales *only its* contribution to the encoder gradient, by comparing the encoder gradient with `loss_classifier` weighted ×1 vs ×3.

In [ ]:
# Reveal:
def encoder_grad(class_weight):
    encoder.zero_grad()
    z = encoder(x)
    (((decoder(z) - x) ** 2).mean() + class_weight * classifier(z).abs().sum()).backward()
    return encoder.weight.grad.clone()

g1 = encoder_grad(1.0)   # = g_decoder + 1·g_class
g3 = encoder_grad(3.0)   # = g_decoder + 3·g_class
implied_g_class = (g3 - g1) / 2.0        # (g3 − g1)/2 should isolate g_class

# Directly measure the classifier branch's OWN pull on the encoder:
encoder.zero_grad(); z = encoder(x); classifier(z).abs().sum().backward()
direct_g_class = encoder.weight.grad.clone()

print("weights change ONLY the weighted component's pull on the shared encoder:")
print(f"  max |(g3 − g1)/2  −  direct g_class| = {(implied_g_class - direct_g_class).abs().max().item():.2e}")
print("  → ~0: the encoder gradient is linear in the component weights (a weighted sum).")

## Section 5 — Common errors

### `RuntimeError: backward through the graph a second time`

You called `.backward()` on more than one component without `retain_graph=True`. The fix isn't `retain_graph` — it's to sum into one `total` and backward *once* (§2.5).

### The encoder isn't learning to classify (only to reconstruct)

The classifier branch's path to the encoder is cut — likely a stray `.detach()` on the latent before the classifier (Exercise 4.1), or `loss_classifier` was dropped from `total`. The encoder only accumulates gradient from components it has a live path to.

### One component dominates the encoder gradient

Not a topology bug — a *scale* bug. Without EMA normalization ([06.4](06.4_the_ema_prior_normalization_deep_dive.ipynb)), the largest-magnitude component swamps the encoder's summed gradient. Normalize *before* summing.

### Separate optimizers per subnetwork fighting each other

If you gave the encoder/decoder/classifier separate optimizers and separate backwards, the shared encoder gets inconsistent updates. Use one optimizer over all parameters and one `total.backward()` — the shared trunk needs the *summed* gradient.

### `zero_grad` in the wrong place

Zeroing gradients *between* the branch backwards (if you insist on separate backwards) erases the accumulation. With the single-`total` approach this can't happen — one backward, then step, then zero. Ordering matters ([05.1](../05_training_loop/05.1_the_custom_training_loop.ipynb)).

## Section 6 — Further reading

- [`src/neural_data_decoding/training/losses/multi_objective.py`](../../src/neural_data_decoding/training/losses/multi_objective.py) — the assembly (`loss_decoder`, `loss_classifier`, `loss_encoder`).
- [02.5 autograd basics](../02_numpy_and_pytorch_basics/02.5_autograd_basics.ipynb) — the graph traversal that makes single-backward routing work.
- [06.1 multi-task losses overview](06.1_multi_task_losses_overview.ipynb) — Critical Note #28, where the single-total-loss rule was stated.

Next up: **[06.12 EMA prior normalization deep dive](06.12_ema_prior_normalization_deep_dive.ipynb)** — the cross-component normalization mechanics, companion to 06.4.